# Caso Práctico - Análisis del Desastre del Titanic con Apache Spark

**Analista de Datos:** Análisis exhaustivo de los pasajeros del Titanic utilizando PySpark (API de DataFrames) y Spark SQL.

Para cada pregunta se resuelve primero con la **API de DataFrames de PySpark** y luego con **Spark SQL**, comparando ambos enfoques, tal como lo exige la guía del caso práctico.

## PARTE 1: Configuración y Exploración Inicial

### 1.1 Configuración del Entorno

In [2]:
# Instalación de PySpark en Google Colab (descomentar si se ejecuta en Colab)
# !pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Crear la sesión de Spark con el nombre "TitanicAnalysis"
spark = SparkSession.builder \
    .appName("TitanicAnalysis") \
    .getOrCreate()

spark

In [5]:
# Cargar el archivo Titanic.csv en un DataFrame, infiriendo el esquema y usando la primera fila como cabecera
df = spark.read.csv("..\\data\\titanic.csv", header=True, inferSchema=True)
df

DataFrame[PassengerId: int, Survived: int, Pclass: int, Name: string, Sex: string, Age: double, SibSp: int, Parch: int, Ticket: string, Fare: double, Cabin: string, Embarked: string]

### 1.2 Exploración Básica

In [7]:
# Mostrar el esquema del DataFrame
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [8]:
# Mostrar las primeras 10 filas
df.show(10)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

In [9]:
# Contar el número total de pasajeros - API de DataFrames
total_pasajeros = df.count()
print(f"Total de pasajeros: {total_pasajeros}")

Total de pasajeros: 891


In [10]:
# Obtener estadísticas descriptivas con describe()
df.describe().show()

+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|summary|      PassengerId|           Survived|            Pclass|                Name|   Sex|               Age|             SibSp|              Parch|            Ticket|             Fare|Cabin|Embarked|
+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|  count|              891|                891|               891|                 891|   891|               714|               891|                891|               891|              891|  204|     889|
|   mean|            446.0| 0.3838383838383838| 2.308641975308642|                NULL|  NULL| 29.69911764705882|0.5230078563411896|0.38159371492704824|260318.54916792738| 32.20420

## PARTE 2: Análisis con Spark SQL - Consultas Básicas

Se registra el DataFrame como vista temporal llamada `Titanic` para poder utilizar Spark SQL.

In [11]:
# Registrar el DataFrame como vista temporal llamada "Titanic"
df.createOrReplaceTempView("Titanic")

### 2.1 ¿Cuántos pasajeros sobrevivieron y cuántos no? (Conteo por Survived)

In [12]:
# --- API de DataFrames ---
# Agrupamos por Survived y contamos pasajeros
df.groupBy("Survived").count().orderBy("Survived").show()

+--------+-----+
|Survived|count|
+--------+-----+
|       0|  549|
|       1|  342|
+--------+-----+



In [13]:
# --- Spark SQL ---
# Conteo de pasajeros agrupado por la columna Survived (0 = No sobrevivió, 1 = Sobrevivió)
spark.sql("""
    SELECT Survived, COUNT(*) AS Total
    FROM Titanic
    GROUP BY Survived
    ORDER BY Survived
""").show()

+--------+-----+
|Survived|Total|
+--------+-----+
|       0|  549|
|       1|  342|
+--------+-----+



### 2.2 ¿Cuál es la distribución de pasajeros por clase (Pclass)?

In [14]:
# --- API de DataFrames ---
df.groupBy("Pclass").count().orderBy("Pclass").show()

+------+-----+
|Pclass|count|
+------+-----+
|     1|  216|
|     2|  184|
|     3|  491|
+------+-----+



In [15]:
# --- Spark SQL ---
# Conteo de pasajeros agrupado por clase del boleto
spark.sql("""
    SELECT Pclass, COUNT(*) AS Total
    FROM Titanic
    GROUP BY Pclass
    ORDER BY Pclass
""").show()

+------+-----+
|Pclass|Total|
+------+-----+
|     1|  216|
|     2|  184|
|     3|  491|
+------+-----+



### 2.3 ¿Cuál es la distribución de pasajeros por sexo?

In [16]:
# --- API de DataFrames ---
df.groupBy("Sex").count().orderBy("Sex").show()

+------+-----+
|   Sex|count|
+------+-----+
|female|  314|
|  male|  577|
+------+-----+



In [17]:
# --- Spark SQL ---
# Conteo de pasajeros agrupado por sexo
spark.sql("""
    SELECT Sex, COUNT(*) AS Total
    FROM Titanic
    GROUP BY Sex
    ORDER BY Sex
""").show()

+------+-----+
|   Sex|Total|
+------+-----+
|female|  314|
|  male|  577|
+------+-----+



### 2.4 ¿Cuál es la distribución de pasajeros por puerto de embarque?

In [18]:
# --- API de DataFrames ---
df.groupBy("Embarked").count().orderBy("Embarked").show()

+--------+-----+
|Embarked|count|
+--------+-----+
|    NULL|    2|
|       C|  168|
|       Q|   77|
|       S|  644|
+--------+-----+



In [19]:
# --- Spark SQL ---
# Conteo de pasajeros agrupado por puerto de embarque (C, Q, S)
spark.sql("""
    SELECT Embarked, COUNT(*) AS Total
    FROM Titanic
    GROUP BY Embarked
    ORDER BY Embarked
""").show()

+--------+-----+
|Embarked|Total|
+--------+-----+
|    NULL|    2|
|       C|  168|
|       Q|   77|
|       S|  644|
+--------+-----+



### 2.5 ¿Cuál es la edad promedio, mínima y máxima de los pasajeros?

In [20]:
# --- API de DataFrames ---
df.select(
    F.avg("Age").alias("Edad_Promedio"),
    F.min("Age").alias("Edad_Minima"),
    F.max("Age").alias("Edad_Maxima")
).show()

+-----------------+-----------+-----------+
|    Edad_Promedio|Edad_Minima|Edad_Maxima|
+-----------------+-----------+-----------+
|29.69911764705882|       0.42|       80.0|
+-----------------+-----------+-----------+



In [21]:
# --- Spark SQL ---
# Estadísticas de la edad de los pasajeros: promedio, mínima y máxima
spark.sql("""
    SELECT
        AVG(Age) AS Edad_Promedio,
        MIN(Age) AS Edad_Minima,
        MAX(Age) AS Edad_Maxima
    FROM Titanic
""").show()

+-----------------+-----------+-----------+
|    Edad_Promedio|Edad_Minima|Edad_Maxima|
+-----------------+-----------+-----------+
|29.69911764705882|       0.42|       80.0|
+-----------------+-----------+-----------+



### 2.6 ¿Cuál es la tarifa promedio, mínima y máxima pagada?

In [22]:
# --- API de DataFrames ---
df.select(
    F.avg("Fare").alias("Tarifa_Promedio"),
    F.min("Fare").alias("Tarifa_Minima"),
    F.max("Fare").alias("Tarifa_Maxima")
).show()

+----------------+-------------+-------------+
| Tarifa_Promedio|Tarifa_Minima|Tarifa_Maxima|
+----------------+-------------+-------------+
|32.2042079685746|          0.0|     512.3292|
+----------------+-------------+-------------+



In [23]:
# --- Spark SQL ---
# Estadísticas de la tarifa pagada: promedio, mínima y máxima
spark.sql("""
    SELECT
        AVG(Fare) AS Tarifa_Promedio,
        MIN(Fare) AS Tarifa_Minima,
        MAX(Fare) AS Tarifa_Maxima
    FROM Titanic
""").show()

+----------------+-------------+-------------+
| Tarifa_Promedio|Tarifa_Minima|Tarifa_Maxima|
+----------------+-------------+-------------+
|32.2042079685746|          0.0|     512.3292|
+----------------+-------------+-------------+



## PARTE 3: Análisis con Spark SQL - Agregaciones por Grupos

### 3.1 ¿Cuál es la tasa de supervivencia por clase?

In [24]:
# --- API de DataFrames ---
df.groupBy("Pclass").agg(
    F.sum("Survived").alias("Sobrevivientes"),
    F.count("*").alias("Total_Pasajeros"),
    (F.sum("Survived") / F.count("*") * 100).alias("Porcentaje_Supervivencia")
).orderBy("Pclass").show()

+------+--------------+---------------+------------------------+
|Pclass|Sobrevivientes|Total_Pasajeros|Porcentaje_Supervivencia|
+------+--------------+---------------+------------------------+
|     1|           136|            216|       62.96296296296296|
|     2|            87|            184|       47.28260869565217|
|     3|           119|            491|      24.236252545824847|
+------+--------------+---------------+------------------------+



In [25]:
# --- Spark SQL ---
# Número de sobrevivientes, total de pasajeros y porcentaje de supervivencia por clase
spark.sql("""
    SELECT
        Pclass,
        SUM(Survived) AS Sobrevivientes,
        COUNT(*) AS Total_Pasajeros,
        ROUND(SUM(Survived) / COUNT(*) * 100, 2) AS Porcentaje_Supervivencia
    FROM Titanic
    GROUP BY Pclass
    ORDER BY Pclass
""").show()

+------+--------------+---------------+------------------------+
|Pclass|Sobrevivientes|Total_Pasajeros|Porcentaje_Supervivencia|
+------+--------------+---------------+------------------------+
|     1|           136|            216|                   62.96|
|     2|            87|            184|                   47.28|
|     3|           119|            491|                   24.24|
+------+--------------+---------------+------------------------+



### 3.2 ¿Cuál es la tasa de supervivencia por sexo?

In [26]:
# --- API de DataFrames ---
df.groupBy("Sex").agg(
    F.sum("Survived").alias("Sobrevivientes"),
    F.count("*").alias("Total_Pasajeros"),
    (F.sum("Survived") / F.count("*") * 100).alias("Porcentaje_Supervivencia")
).orderBy("Sex").show()

+------+--------------+---------------+------------------------+
|   Sex|Sobrevivientes|Total_Pasajeros|Porcentaje_Supervivencia|
+------+--------------+---------------+------------------------+
|female|           233|            314|       74.20382165605095|
|  male|           109|            577|      18.890814558058924|
+------+--------------+---------------+------------------------+



In [27]:
# --- Spark SQL ---
# Número de sobrevivientes, total de pasajeros y porcentaje de supervivencia por sexo
spark.sql("""
    SELECT
        Sex,
        SUM(Survived) AS Sobrevivientes,
        COUNT(*) AS Total_Pasajeros,
        ROUND(SUM(Survived) / COUNT(*) * 100, 2) AS Porcentaje_Supervivencia
    FROM Titanic
    GROUP BY Sex
    ORDER BY Sex
""").show()

+------+--------------+---------------+------------------------+
|   Sex|Sobrevivientes|Total_Pasajeros|Porcentaje_Supervivencia|
+------+--------------+---------------+------------------------+
|female|           233|            314|                    74.2|
|  male|           109|            577|                   18.89|
+------+--------------+---------------+------------------------+



### 3.3 ¿Cuál es la edad promedio por clase y estado de supervivencia?

In [28]:
# --- API de DataFrames ---
df.groupBy("Pclass", "Survived").agg(
    F.avg("Age").alias("Edad_Promedio")
).orderBy("Pclass", "Survived").show()

+------+--------+------------------+
|Pclass|Survived|     Edad_Promedio|
+------+--------+------------------+
|     1|       0|        43.6953125|
|     1|       1| 35.36819672131148|
|     2|       0|33.544444444444444|
|     2|       1| 25.90156626506024|
|     3|       0|26.555555555555557|
|     3|       1|20.646117647058823|
+------+--------+------------------+



In [29]:
# --- Spark SQL ---
# Edad promedio agrupando por clase (Pclass) y estado de supervivencia (Survived)
spark.sql("""
    SELECT
        Pclass,
        Survived,
        ROUND(AVG(Age), 2) AS Edad_Promedio
    FROM Titanic
    GROUP BY Pclass, Survived
    ORDER BY Pclass, Survived
""").show()

+------+--------+-------------+
|Pclass|Survived|Edad_Promedio|
+------+--------+-------------+
|     1|       0|         43.7|
|     1|       1|        35.37|
|     2|       0|        33.54|
|     2|       1|         25.9|
|     3|       0|        26.56|
|     3|       1|        20.65|
+------+--------+-------------+



### 3.4 ¿Cuál es la tarifa promedio pagada por clase?

In [30]:
# --- API de DataFrames ---
df.groupBy("Pclass").agg(
    F.avg("Fare").alias("Tarifa_Promedio")
).orderBy("Pclass").show()

+------+------------------+
|Pclass|   Tarifa_Promedio|
+------+------------------+
|     1| 84.15468749999992|
|     2| 20.66218315217391|
|     3|13.675550101832997|
+------+------------------+



In [31]:
# --- Spark SQL ---
# Tarifa promedio pagada, agrupando por clase
spark.sql("""
    SELECT
        Pclass,
        ROUND(AVG(Fare), 2) AS Tarifa_Promedio
    FROM Titanic
    GROUP BY Pclass
    ORDER BY Pclass
""").show()

+------+---------------+
|Pclass|Tarifa_Promedio|
+------+---------------+
|     1|          84.15|
|     2|          20.66|
|     3|          13.68|
+------+---------------+



### 3.5 ¿Cuál es la distribución de supervivencia por puerto de embarque?

In [32]:
# --- API de DataFrames ---
df.groupBy("Embarked").agg(
    F.sum(F.when(F.col("Survived") == 1, 1).otherwise(0)).alias("Sobrevivientes"),
    F.sum(F.when(F.col("Survived") == 0, 1).otherwise(0)).alias("No_Sobrevivientes")
).orderBy("Embarked").show()

+--------+--------------+-----------------+
|Embarked|Sobrevivientes|No_Sobrevivientes|
+--------+--------------+-----------------+
|    NULL|             2|                0|
|       C|            93|               75|
|       Q|            30|               47|
|       S|           217|              427|
+--------+--------------+-----------------+



In [33]:
# --- Spark SQL ---
# Número de sobrevivientes y no sobrevivientes agrupando por puerto de embarque
spark.sql("""
    SELECT
        Embarked,
        SUM(CASE WHEN Survived = 1 THEN 1 ELSE 0 END) AS Sobrevivientes,
        SUM(CASE WHEN Survived = 0 THEN 1 ELSE 0 END) AS No_Sobrevivientes
    FROM Titanic
    GROUP BY Embarked
    ORDER BY Embarked
""").show()

+--------+--------------+-----------------+
|Embarked|Sobrevivientes|No_Sobrevivientes|
+--------+--------------+-----------------+
|    NULL|             2|                0|
|       C|            93|               75|
|       Q|            30|               47|
|       S|           217|              427|
+--------+--------------+-----------------+



### 3.6 ¿Cuál es el número promedio de familiares a bordo por clase?

Se crea la columna calculada `Familiares = SibSp + Parch` y se agrupa por clase.

In [34]:
# --- API de DataFrames ---
df_familiares = df.withColumn("Familiares", F.col("SibSp") + F.col("Parch"))
df_familiares.groupBy("Pclass").agg(
    F.avg("Familiares").alias("Promedio_Familiares")
).orderBy("Pclass").show()

+------+-------------------+
|Pclass|Promedio_Familiares|
+------+-------------------+
|     1| 0.7731481481481481|
|     2|  0.782608695652174|
|     3| 1.0081466395112015|
+------+-------------------+



In [35]:
# --- Spark SQL ---
# Se crea la columna Familiares = SibSp + Parch dentro de la propia consulta
# y se calcula el promedio de familiares por clase
spark.sql("""
    SELECT
        Pclass,
        ROUND(AVG(SibSp + Parch), 2) AS Promedio_Familiares
    FROM Titanic
    GROUP BY Pclass
    ORDER BY Pclass
""").show()

+------+-------------------+
|Pclass|Promedio_Familiares|
+------+-------------------+
|     1|               0.77|
|     2|               0.78|
|     3|               1.01|
+------+-------------------+



## PARTE 4: Análisis con Spark SQL - Consultas Avanzadas

### 4.1 DataFrame que muestre solo los pasajeros que sobrevivieron (Survived = 1)

In [36]:
# --- API de DataFrames ---
df_sobrevivientes = df.filter(F.col("Survived") == 1)
df_sobrevivientes.show(10)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          9|       1|     3|Johnson, Mrs. Osc...|female|27.0|    0|    2|          347742|11.1333| NULL|       S|
|         10|       1|     2|Nasser, Mrs. Nich...|female|14.0|    1|    0|          237736|30.0708| NULL|       C|
|         11|       1|     3|Sandstrom, Miss. ...|female| 4.0|    1|    1|      

In [37]:
# --- Spark SQL ---
df_sobrevivientes_sql = spark.sql("""
    SELECT *
    FROM Titanic
    WHERE Survived = 1
""")
df_sobrevivientes_sql.show(10)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          9|       1|     3|Johnson, Mrs. Osc...|female|27.0|    0|    2|          347742|11.1333| NULL|       S|
|         10|       1|     2|Nasser, Mrs. Nich...|female|14.0|    1|    0|          237736|30.0708| NULL|       C|
|         11|       1|     3|Sandstrom, Miss. ...|female| 4.0|    1|    1|      

### 4.2 DataFrame con solo pasajeros de sexo femenino (Sex = 'female')

In [38]:
# --- API de DataFrames ---
df_mujeres = df.filter(F.col("Sex") == "female")
df_mujeres.show(10)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          9|       1|     3|Johnson, Mrs. Osc...|female|27.0|    0|    2|          347742|11.1333| NULL|       S|
|         10|       1|     2|Nasser, Mrs. Nich...|female|14.0|    1|    0|          237736|30.0708| NULL|       C|
|         11|       1|     3|Sandstrom, Miss. ...|female| 4.0|    1|    1|      

In [39]:
# --- Spark SQL ---
df_mujeres_sql = spark.sql("""
    SELECT *
    FROM Titanic
    WHERE Sex = 'female'
""")
df_mujeres_sql.show(10)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          9|       1|     3|Johnson, Mrs. Osc...|female|27.0|    0|    2|          347742|11.1333| NULL|       S|
|         10|       1|     2|Nasser, Mrs. Nich...|female|14.0|    1|    0|          237736|30.0708| NULL|       C|
|         11|       1|     3|Sandstrom, Miss. ...|female| 4.0|    1|    1|      

### 4.3 DataFrame con pasajeros de primera clase que sobrevivieron

In [40]:
# --- API de DataFrames ---
df_primera_sobrevivientes = df.filter((F.col("Pclass") == 1) & (F.col("Survived") == 1))
df_primera_sobrevivientes.show(10)

+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|    Fare|      Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----------+--------+
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|PC 17599| 71.2833|        C85|       C|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|  113803|    53.1|       C123|       S|
|         12|       1|     1|Bonnell, Miss. El...|female|58.0|    0|    0|  113783|   26.55|       C103|       S|
|         24|       1|     1|Sloper, Mr. Willi...|  male|28.0|    0|    0|  113788|    35.5|         A6|       S|
|         32|       1|     1|Spencer, Mrs. Wil...|female|NULL|    1|    0|PC 17569|146.5208|        B78|       C|
|         53|       1|     1|Harper, Mrs. Henr...|female|49.0|    1|    0|PC 17572| 76.7

In [41]:
# --- Spark SQL ---
df_primera_sobrevivientes_sql = spark.sql("""
    SELECT *
    FROM Titanic
    WHERE Pclass = 1 AND Survived = 1
""")
df_primera_sobrevivientes_sql.show(10)

+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|    Fare|      Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----------+--------+
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|PC 17599| 71.2833|        C85|       C|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|  113803|    53.1|       C123|       S|
|         12|       1|     1|Bonnell, Miss. El...|female|58.0|    0|    0|  113783|   26.55|       C103|       S|
|         24|       1|     1|Sloper, Mr. Willi...|  male|28.0|    0|    0|  113788|    35.5|         A6|       S|
|         32|       1|     1|Spencer, Mrs. Wil...|female|NULL|    1|    0|PC 17569|146.5208|        B78|       C|
|         53|       1|     1|Harper, Mrs. Henr...|female|49.0|    1|    0|PC 17572| 76.7

### 4.4 DataFrame sin las columnas Ticket, Fare, Cabin, Embarked

In [42]:
# --- API de DataFrames ---
df_sin_columnas = df.drop("Ticket", "Fare", "Cabin", "Embarked")
df_sin_columnas.show(10)

+-----------+--------+------+--------------------+------+----+-----+-----+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|
+-----------+--------+------+--------------------+------+----+-----+-----+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|
|          7|       0|     1|McCarthy, Mr. Tim...|  male|54.0|    0|    0|
|          8|       0|     3|Palsson, Master. ...|  male| 2.0|    3|    1|
|          9|       1|     3|Johnson, Mrs. Osc...|female|27.0|    0|    2|
|         10|       1|     2|Nasser, Mrs. Nich...|female|14.0|    1|    0|
+-----------+--------+---

In [43]:
# --- Spark SQL ---
# Se seleccionan explícitamente solo las columnas que se desean conservar
df_sin_columnas_sql = spark.sql("""
    SELECT PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch
    FROM Titanic
""")
df_sin_columnas_sql.show(10)

+-----------+--------+------+--------------------+------+----+-----+-----+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|
+-----------+--------+------+--------------------+------+----+-----+-----+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|
|          7|       0|     1|McCarthy, Mr. Tim...|  male|54.0|    0|    0|
|          8|       0|     3|Palsson, Master. ...|  male| 2.0|    3|    1|
|          9|       1|     3|Johnson, Mrs. Osc...|female|27.0|    0|    2|
|         10|       1|     2|Nasser, Mrs. Nich...|female|14.0|    1|    0|
+-----------+--------+---

### 4.5 DataFrame con pasajeros ordenados por edad de mayor a menor

In [40]:
# --- API de DataFrames ---
df_orden_edad = df.orderBy(F.col("Age").desc())
df_orden_edad.show(10)

+-----------+--------+------+--------------------+----+----+-----+-----+----------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name| Sex| Age|SibSp|Parch|    Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+----+----+-----+-----+----------+-------+-----+--------+
|        631|       1|     1|Barkworth, Mr. Al...|male|80.0|    0|    0|     27042|   30.0|  A23|       S|
|        852|       0|     3| Svensson, Mr. Johan|male|74.0|    0|    0|    347060|  7.775| NULL|       S|
|         97|       0|     1|Goldschmidt, Mr. ...|male|71.0|    0|    0|  PC 17754|34.6542|   A5|       C|
|        494|       0|     1|Artagaveytia, Mr....|male|71.0|    0|    0|  PC 17609|49.5042| NULL|       C|
|        117|       0|     3|Connors, Mr. Patrick|male|70.5|    0|    0|    370369|   7.75| NULL|       Q|
|        673|       0|     2|Mitchell, Mr. Hen...|male|70.0|    0|    0|C.A. 24580|   10.5| NULL|       S|
|        746|       0|     1|Crosby, 

In [41]:
# --- Spark SQL ---
df_orden_edad_sql = spark.sql("""
    SELECT *
    FROM Titanic
    ORDER BY Age DESC
""")
df_orden_edad_sql.show(10)

+-----------+--------+------+--------------------+----+----+-----+-----+----------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name| Sex| Age|SibSp|Parch|    Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+----+----+-----+-----+----------+-------+-----+--------+
|        631|       1|     1|Barkworth, Mr. Al...|male|80.0|    0|    0|     27042|   30.0|  A23|       S|
|        852|       0|     3| Svensson, Mr. Johan|male|74.0|    0|    0|    347060|  7.775| NULL|       S|
|         97|       0|     1|Goldschmidt, Mr. ...|male|71.0|    0|    0|  PC 17754|34.6542|   A5|       C|
|        494|       0|     1|Artagaveytia, Mr....|male|71.0|    0|    0|  PC 17609|49.5042| NULL|       C|
|        117|       0|     3|Connors, Mr. Patrick|male|70.5|    0|    0|    370369|   7.75| NULL|       Q|
|        673|       0|     2|Mitchell, Mr. Hen...|male|70.0|    0|    0|C.A. 24580|   10.5| NULL|       S|
|        746|       0|     1|Crosby, 

### 4.6 DataFrame con los 10 pasajeros que pagaron las tarifas más altas

In [44]:
# --- API de DataFrames ---
df.orderBy(F.col("Fare").desc()).limit(10).show()

+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+---------------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|    Fare|          Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+---------------+--------+
|        680|       1|     1|Cardeza, Mr. Thom...|  male|36.0|    0|    1|PC 17755|512.3292|    B51 B53 B55|       C|
|        259|       1|     1|    Ward, Miss. Anna|female|35.0|    0|    0|PC 17755|512.3292|           NULL|       C|
|        738|       1|     1|Lesurer, Mr. Gust...|  male|35.0|    0|    0|PC 17755|512.3292|           B101|       C|
|         89|       1|     1|Fortune, Miss. Ma...|female|23.0|    3|    2|   19950|   263.0|    C23 C25 C27|       S|
|         28|       0|     1|Fortune, Mr. Char...|  male|19.0|    3|    2|   19950|   263.0|    C23 C25 C27|       S|
|        342|       1|     1|Fortune, Miss. Al...|female

In [46]:
# --- Spark SQL ---
spark.sql("""
    SELECT *
    FROM Titanic
    ORDER BY Fare DESC
    LIMIT 10
""").show()

+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+---------------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|    Fare|          Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+---------------+--------+
|        680|       1|     1|Cardeza, Mr. Thom...|  male|36.0|    0|    1|PC 17755|512.3292|    B51 B53 B55|       C|
|        259|       1|     1|    Ward, Miss. Anna|female|35.0|    0|    0|PC 17755|512.3292|           NULL|       C|
|        738|       1|     1|Lesurer, Mr. Gust...|  male|35.0|    0|    0|PC 17755|512.3292|           B101|       C|
|         89|       1|     1|Fortune, Miss. Ma...|female|23.0|    3|    2|   19950|   263.0|    C23 C25 C27|       S|
|         28|       0|     1|Fortune, Mr. Char...|  male|19.0|    3|    2|   19950|   263.0|    C23 C25 C27|       S|
|        342|       1|     1|Fortune, Miss. Al...|female

### 4.7 Columna Grupo_Edad usando CASE WHEN

Categorías: "Niño" (Age < 12), "Adolescente" (12 a 17), "Adulto" (18 a 59), "Adulto Mayor" (>= 60), "Desconocido" (NULL).

In [44]:
# --- API de DataFrames (equivalente a CASE WHEN con la función when/otherwise) ---
df_grupo_edad = df.withColumn(
    "Grupo_Edad",
    F.when(F.col("Age").isNull(), "Desconocido")
     .when(F.col("Age") < 12, "Niño")
     .when((F.col("Age") >= 12) & (F.col("Age") <= 17), "Adolescente")
     .when((F.col("Age") >= 18) & (F.col("Age") <= 59), "Adulto")
     .otherwise("Adulto Mayor")
)
df_grupo_edad.select("PassengerId", "Age", "Grupo_Edad").show(10)

+-----------+----+-----------+
|PassengerId| Age| Grupo_Edad|
+-----------+----+-----------+
|          1|22.0|     Adulto|
|          2|38.0|     Adulto|
|          3|26.0|     Adulto|
|          4|35.0|     Adulto|
|          5|35.0|     Adulto|
|          6|NULL|Desconocido|
|          7|54.0|     Adulto|
|          8| 2.0|       Niño|
|          9|27.0|     Adulto|
|         10|14.0|Adolescente|
+-----------+----+-----------+
only showing top 10 rows


In [47]:
# --- Spark SQL ---
# Se utiliza CASE WHEN para categorizar la edad en grupos
df_grupo_edad_sql = spark.sql("""
    SELECT
        *,
        CASE
            WHEN Age IS NULL THEN 'Desconocido'
            WHEN Age < 12 THEN 'Niño'
            WHEN Age BETWEEN 12 AND 17 THEN 'Adolescente'
            WHEN Age BETWEEN 18 AND 59 THEN 'Adulto'
            WHEN Age >= 60 THEN 'Adulto Mayor'
        END AS Grupo_Edad
    FROM Titanic
""")
df_grupo_edad_sql.select("PassengerId", "Age", "Grupo_Edad").show(10)

# Registramos esta vista para reutilizarla en las siguientes consultas (4.8)
df_grupo_edad_sql.createOrReplaceTempView("TitanicGrupoEdad")

+-----------+----+-----------+
|PassengerId| Age| Grupo_Edad|
+-----------+----+-----------+
|          1|22.0|     Adulto|
|          2|38.0|     Adulto|
|          3|26.0|     Adulto|
|          4|35.0|     Adulto|
|          5|35.0|     Adulto|
|          6|NULL|Desconocido|
|          7|54.0|     Adulto|
|          8| 2.0|       Niño|
|          9|27.0|     Adulto|
|         10|14.0|Adolescente|
+-----------+----+-----------+
only showing top 10 rows


### 4.8 Tasa de supervivencia por grupo de edad

In [ ]:
# --- API de DataFrames ---
df_grupo_edad.groupBy("Grupo_Edad").agg(
    F.sum("Survived").alias("Sobrevivientes"),
    F.count("*").alias("Total_Pasajeros"),
    (F.sum("Survived") / F.count("*") * 100).alias("Porcentaje_Supervivencia")
).orderBy("Grupo_Edad").show()

In [49]:
# --- Spark SQL ---
# Se reutiliza la vista TitanicGrupoEdad creada en el punto 4.7
spark.sql("""
    SELECT
        Grupo_Edad,
        SUM(Survived) AS Sobrevivientes,
        COUNT(*) AS Total_Pasajeros,
        ROUND(SUM(Survived) / COUNT(*) * 100, 2) AS Porcentaje_Supervivencia
    FROM TitanicGrupoEdad
    GROUP BY Grupo_Edad
    ORDER BY Grupo_Edad
""").show()

+------------+--------------+---------------+------------------------+
|  Grupo_Edad|Sobrevivientes|Total_Pasajeros|Porcentaje_Supervivencia|
+------------+--------------+---------------+------------------------+
| Adolescente|            22|             45|                   48.89|
|      Adulto|           222|            575|                   38.61|
|Adulto Mayor|             7|             26|                   26.92|
| Desconocido|            52|            177|                   29.38|
|        Niño|            39|             68|                   57.35|
+------------+--------------+---------------+------------------------+



### 4.9 Tarifa total recaudada por clase

In [50]:
# --- API de DataFrames ---
df.groupBy("Pclass").agg(
    F.sum("Fare").alias("Tarifa_Total")
).orderBy("Pclass").show()

+------+------------------+
|Pclass|      Tarifa_Total|
+------+------------------+
|     1|18177.412499999984|
|     2|3801.8416999999995|
|     3| 6714.695100000002|
+------+------------------+



In [49]:
# --- Spark SQL ---
spark.sql("""
    SELECT
        Pclass,
        ROUND(SUM(Fare), 2) AS Tarifa_Total
    FROM Titanic
    GROUP BY Pclass
    ORDER BY Pclass
""").show()

+------+------------+
|Pclass|Tarifa_Total|
+------+------------+
|     1|    18177.41|
|     2|     3801.84|
|     3|      6714.7|
+------+------------+



### 4.10 Clases donde la edad promedio de los sobrevivientes es mayor a 30 años (HAVING / subconsulta)

In [51]:
# --- API de DataFrames ---
df.filter(F.col("Survived") == 1) \
  .groupBy("Pclass") \
  .agg(F.avg("Age").alias("Edad_Promedio_Sobrevivientes")) \
  .filter(F.col("Edad_Promedio_Sobrevivientes") > 30) \
  .orderBy("Pclass") \
  .show()

+------+----------------------------+
|Pclass|Edad_Promedio_Sobrevivientes|
+------+----------------------------+
|     1|           35.36819672131148|
+------+----------------------------+



In [52]:
# --- Spark SQL con HAVING ---
spark.sql("""
    SELECT
        Pclass,
        ROUND(AVG(Age), 2) AS Edad_Promedio_Sobrevivientes
    FROM Titanic
    WHERE Survived = 1
    GROUP BY Pclass
    HAVING AVG(Age) > 30
    ORDER BY Pclass
""").show()

+------+----------------------------+
|Pclass|Edad_Promedio_Sobrevivientes|
+------+----------------------------+
|     1|                       35.37|
+------+----------------------------+



In [53]:
# --- Spark SQL con subconsulta (equivalente usando una subquery en el FROM) ---
spark.sql("""
    SELECT Pclass, Edad_Promedio_Sobrevivientes
    FROM (
        SELECT
            Pclass,
            AVG(Age) AS Edad_Promedio_Sobrevivientes
        FROM Titanic
        WHERE Survived = 1
        GROUP BY Pclass
    ) AS Promedios
    WHERE Edad_Promedio_Sobrevivientes > 30
    ORDER BY Pclass
""").show()

+------+----------------------------+
|Pclass|Edad_Promedio_Sobrevivientes|
+------+----------------------------+
|     1|           35.36819672131148|
+------+----------------------------+



## PARTE 5: Limpieza de Datos y Transformaciones

### 5.1 ¿Cuántos valores nulos hay en la columna Age?

In [54]:
# --- API de DataFrames ---
nulos_age = df.filter(F.col("Age").isNull()).count()
print(f"Valores nulos en Age: {nulos_age}")

Valores nulos en Age: 177


In [56]:
# --- Spark SQL ---
spark.sql("""
    SELECT COUNT(*) AS Nulos_Age
    FROM Titanic
    WHERE Age IS NULL
""").show()

+---------+
|Nulos_Age|
+---------+
|      177|
+---------+



### 5.2 Reemplazar los valores nulos en Age por la edad promedio de todos los pasajeros

In [55]:
# --- API de DataFrames ---
edad_promedio = df.select(F.avg("Age")).collect()[0][0]
df_age_imputada = df.fillna({"Age": edad_promedio})
df_age_imputada.select("PassengerId", "Age").show(10)
print(f"Edad promedio utilizada para la imputación: {edad_promedio:.2f}")

+-----------+-----------------+
|PassengerId|              Age|
+-----------+-----------------+
|          1|             22.0|
|          2|             38.0|
|          3|             26.0|
|          4|             35.0|
|          5|             35.0|
|          6|29.69911764705882|
|          7|             54.0|
|          8|              2.0|
|          9|             27.0|
|         10|             14.0|
+-----------+-----------------+
only showing top 10 rows
Edad promedio utilizada para la imputación: 29.70


In [58]:
# --- Spark SQL ---
# Se calcula el promedio con una subconsulta y se usa COALESCE para reemplazar los nulos
df_age_imputada_sql = spark.sql("""
    SELECT
        *,
        COALESCE(Age, (SELECT AVG(Age) FROM Titanic)) AS Age_Imputada
    FROM Titanic
""")
df_age_imputada_sql.select("PassengerId", "Age", "Age_Imputada").show(10)

+-----------+----+-----------------+
|PassengerId| Age|     Age_Imputada|
+-----------+----+-----------------+
|          1|22.0|             22.0|
|          2|38.0|             38.0|
|          3|26.0|             26.0|
|          4|35.0|             35.0|
|          5|35.0|             35.0|
|          6|NULL|29.69911764705882|
|          7|54.0|             54.0|
|          8| 2.0|              2.0|
|          9|27.0|             27.0|
|         10|14.0|             14.0|
+-----------+----+-----------------+
only showing top 10 rows


### 5.3 Columna calculada Descripcion_Sexo

"male" → "Masculino", "female" → "Femenino".

In [59]:
# --- API de DataFrames ---
df_descripcion_sexo = df.withColumn(
    "Descripcion_Sexo",
    F.when(F.col("Sex") == "male", "Masculino")
     .when(F.col("Sex") == "female", "Femenino")
)
df_descripcion_sexo.select("PassengerId", "Sex", "Descripcion_Sexo").show(10)

+-----------+------+----------------+
|PassengerId|   Sex|Descripcion_Sexo|
+-----------+------+----------------+
|          1|  male|       Masculino|
|          2|female|        Femenino|
|          3|female|        Femenino|
|          4|female|        Femenino|
|          5|  male|       Masculino|
|          6|  male|       Masculino|
|          7|  male|       Masculino|
|          8|  male|       Masculino|
|          9|female|        Femenino|
|         10|female|        Femenino|
+-----------+------+----------------+
only showing top 10 rows


In [58]:
# --- Spark SQL ---
df_descripcion_sexo_sql = spark.sql("""
    SELECT
        *,
        CASE
            WHEN Sex = 'male' THEN 'Masculino'
            WHEN Sex = 'female' THEN 'Femenino'
        END AS Descripcion_Sexo
    FROM Titanic
""")
df_descripcion_sexo_sql.select("PassengerId", "Sex", "Descripcion_Sexo").show(10)

+-----------+------+----------------+
|PassengerId|   Sex|Descripcion_Sexo|
+-----------+------+----------------+
|          1|  male|       Masculino|
|          2|female|        Femenino|
|          3|female|        Femenino|
|          4|female|        Femenino|
|          5|  male|       Masculino|
|          6|  male|       Masculino|
|          7|  male|       Masculino|
|          8|  male|       Masculino|
|          9|female|        Femenino|
|         10|female|        Femenino|
+-----------+------+----------------+
only showing top 10 rows


### 5.4 Columna calculada Clase_Descripcion

1 → "Primera Clase", 2 → "Segunda Clase", 3 → "Tercera Clase".

In [59]:
# --- API de DataFrames ---
df_clase_descripcion = df.withColumn(
    "Clase_Descripcion",
    F.when(F.col("Pclass") == 1, "Primera Clase")
     .when(F.col("Pclass") == 2, "Segunda Clase")
     .when(F.col("Pclass") == 3, "Tercera Clase")
)
df_clase_descripcion.select("PassengerId", "Pclass", "Clase_Descripcion").show(10)

+-----------+------+-----------------+
|PassengerId|Pclass|Clase_Descripcion|
+-----------+------+-----------------+
|          1|     3|    Tercera Clase|
|          2|     1|    Primera Clase|
|          3|     3|    Tercera Clase|
|          4|     1|    Primera Clase|
|          5|     3|    Tercera Clase|
|          6|     3|    Tercera Clase|
|          7|     1|    Primera Clase|
|          8|     3|    Tercera Clase|
|          9|     3|    Tercera Clase|
|         10|     2|    Segunda Clase|
+-----------+------+-----------------+
only showing top 10 rows


In [60]:
# --- Spark SQL ---
df_clase_descripcion_sql = spark.sql("""
    SELECT
        *,
        CASE
            WHEN Pclass = 1 THEN 'Primera Clase'
            WHEN Pclass = 2 THEN 'Segunda Clase'
            WHEN Pclass = 3 THEN 'Tercera Clase'
        END AS Clase_Descripcion
    FROM Titanic
""")
df_clase_descripcion_sql.select("PassengerId", "Pclass", "Clase_Descripcion").show(10)

# Registramos esta vista con las descripciones para usarla más adelante si se requiere
df_clase_descripcion_sql.createOrReplaceTempView("TitanicClaseDescripcion")

+-----------+------+-----------------+
|PassengerId|Pclass|Clase_Descripcion|
+-----------+------+-----------------+
|          1|     3|    Tercera Clase|
|          2|     1|    Primera Clase|
|          3|     3|    Tercera Clase|
|          4|     1|    Primera Clase|
|          5|     3|    Tercera Clase|
|          6|     3|    Tercera Clase|
|          7|     1|    Primera Clase|
|          8|     3|    Tercera Clase|
|          9|     3|    Tercera Clase|
|         10|     2|    Segunda Clase|
+-----------+------+-----------------+
only showing top 10 rows


## PARTE 6: Análisis Multidimensional con Spark SQL

### 6.1 Edad promedio de sobrevivientes vs. no sobrevivientes, por sexo

In [61]:
# --- API de DataFrames ---
df.groupBy("Sex", "Survived").agg(
    F.avg("Age").alias("Edad_Promedio")
).orderBy("Sex", "Survived").show()

+------+--------+------------------+
|   Sex|Survived|     Edad_Promedio|
+------+--------+------------------+
|female|       0|         25.046875|
|female|       1| 28.84771573604061|
|  male|       0|31.618055555555557|
|  male|       1|27.276021505376345|
+------+--------+------------------+



In [62]:
# --- Spark SQL ---
spark.sql("""
    SELECT
        Sex,
        Survived,
        ROUND(AVG(Age), 2) AS Edad_Promedio
    FROM Titanic
    GROUP BY Sex, Survived
    ORDER BY Sex, Survived
""").show()

+------+--------+-------------+
|   Sex|Survived|Edad_Promedio|
+------+--------+-------------+
|female|       0|        25.05|
|female|       1|        28.85|
|  male|       0|        31.62|
|  male|       1|        27.28|
+------+--------+-------------+



### 6.2 Tarifa promedio pagada por sobrevivientes vs. no sobrevivientes, por clase

In [63]:
# --- API de DataFrames ---
df.groupBy("Pclass", "Survived").agg(
    F.avg("Fare").alias("Tarifa_Promedio")
).orderBy("Pclass", "Survived").show()

+------+--------+------------------+
|Pclass|Survived|   Tarifa_Promedio|
+------+--------+------------------+
|     1|       0| 64.68400750000002|
|     1|       1| 95.60802867647055|
|     2|       0|19.412327835051546|
|     2|       1|           22.0557|
|     3|       0| 13.66936424731183|
|     3|       1|13.694887394957966|
+------+--------+------------------+



In [64]:
# --- Spark SQL ---
spark.sql("""
    SELECT
        Pclass,
        Survived,
        ROUND(AVG(Fare), 2) AS Tarifa_Promedio
    FROM Titanic
    GROUP BY Pclass, Survived
    ORDER BY Pclass, Survived
""").show()

+------+--------+---------------+
|Pclass|Survived|Tarifa_Promedio|
+------+--------+---------------+
|     1|       0|          64.68|
|     1|       1|          95.61|
|     2|       0|          19.41|
|     2|       1|          22.06|
|     3|       0|          13.67|
|     3|       1|          13.69|
+------+--------+---------------+



## Conclusiones

- La tasa de supervivencia fue notablemente mayor en primera clase y entre las mujeres, confirmando el patrón histórico de "mujeres y niños primero" y el trato preferencial hacia los pasajeros de clases más altas.
- La columna `Age` presenta valores nulos que fueron imputados con la media para no perder registros en los análisis posteriores.
- Se combinaron con éxito la API de DataFrames de PySpark y Spark SQL para resolver todas las consultas solicitadas, incluyendo agregaciones, filtros, `CASE WHEN`, subconsultas y `HAVING`.

In [60]:
# Finalizar la sesión de Spark
spark.stop()